# 4.2

# xml: html처럼 태그 기반으로 자료를 저장한 포멧
* xml parser를 통해서 str을 xml로 변환하는 작업필요
* xml로 변환이 되면 태그 기반으로 자료를 찾아서 정리
* 태그에서 자료를 추출할 때는 beautifulsoup이라는 라이브러리를 이용

# 전세대출 데이터 수집하기

In [4]:
import requests
import json
import os
import time
import pandas as pd

In [4]:
url="http://apis.data.go.kr/B551408/rent-loan-rate-info/rate-list"
servicekey="TUszyF9E+ahKSr7EHb1uL3WyU458jqCE99oqTvyRDyQ9+oN7KcVAIijXzvc8iFsJhgTjoKHuRWT2wV35QeNBZw=="
payload=dict(serviceKey=servicekey, numOfRows=10, pageNo=1, dataType='JSON')
r=requests.get(url, params=payload)
response=r.json()
response

{'header': {'resultCode': '00', 'resultMsg': '정상'},
 'body': {'pageNo': 1,
  'totalCount': 18,
  'numOfRows': 10,
  'items': [{'bssYmdStart': '20250526',
    'interest4_1': '0',
    'interest3_2': '0',
    'interest4_2': '0',
    'interest2_1': '0',
    'interest1_2': '0',
    'interest3_1': '0',
    'interest2_2': '0',
    'interest1_1': '0',
    'bssYmdEnd': '20250601',
    'organId': '산업은행',
    'callCenter': '1588-1500'},
   {'bssYmdStart': '20250526',
    'interest4_1': '0',
    'interest3_2': '0',
    'interest4_2': '0',
    'interest2_1': '0',
    'interest1_2': '0',
    'interest3_1': '0',
    'interest2_2': '0',
    'interest1_1': '0',
    'bssYmdEnd': '20250601',
    'organId': '수협은행',
    'callCenter': '1588-1515'},
   {'bssYmdStart': '20250526',
    'interest4_1': '0',
    'interest3_2': '0',
    'interest4_2': '0',
    'interest2_1': '0',
    'interest1_2': '0',
    'interest3_1': '0',
    'interest2_2': '0',
    'interest1_1': '0',
    'bssYmdEnd': '20250601',
    'organI

In [9]:
def unpack_data(response):
    result={}
    for item in response['body']['items']:
        for key, value in item.items():
            result.setdefault(key,[]).append(value)
    df=pd.DataFrame(result)
    return df

,bssYmdStart,interest4_1,interest3_2,interest4_2,interest2_1,interest1_2,interest3_1,interest2_2,interest1_1,bssYmdEnd,organId,callCenter
0,20250526,0,0,0,0,0,0,0,0,20250601,산업은행,1588-1500
1,20250526,0,0,0,0,0,0,0,0,20250601,수협은행,1588-1515
2,20250526,0,0,0,0,0,0,0,0,20250601,SC은행,1588-1599
3,20250526,5.06,0,0,0,0,0,0,0,20250601,경남은행,1588-8585
4,20250526,0,0,0,0,0,0,0,0,20250601,제주은행,1588-0079
5,20250526,4.04,0,0,0,0,0,0,0,20250601,광주은행,1600-4000
6,20250526,4.8,0,0,0,0,0,0,0,20250601,전북은행,1588-4477
7,20250526,3.37,0,0,0,0,0,0,0,20250601,토스뱅크,1661-7654
8,20250526,2.92,0,0,0,0,0,0,0,20250601,아이엠뱅크,1588-5050
9,20250526,3.93,0,0,0,0,0,0,0,20250601,우리은행,1599-5000


In [14]:
page=1
result={}
while True:
    url="http://apis.data.go.kr/B551408/rent-loan-rate-info/rate-list"
    servicekey="TUszyF9E+ahKSr7EHb1uL3WyU458jqCE99oqTvyRDyQ9+oN7KcVAIijXzvc8iFsJhgTjoKHuRWT2wV35QeNBZw=="
    payload=dict(serviceKey=servicekey, numOfRows=10, pageNo=page, dataType='JSON')
    r=requests.get(url, params=payload)
    response=r.json()
    for item in response['body']['items']:
        for key, value in item.items():
            result.setdefault(key,[]).append(value)
    total_page=(response['body']['totalCount']//10)+1
    if page<total_page:
        page+=1
    else:
        break
df=pd.DataFrame(result)
df

,bssYmdStart,interest4_1,interest3_2,interest4_2,interest2_1,interest1_2,interest3_1,interest2_2,interest1_1,bssYmdEnd,organId,callCenter
0,20250526,0,0,0,0,0,0,0,0,20250601,산업은행,1588-1500
1,20250526,0,0,0,0,0,0,0,0,20250601,수협은행,1588-1515
2,20250526,0,0,0,0,0,0,0,0,20250601,SC은행,1588-1599
3,20250526,5.06,0,0,0,0,0,0,0,20250601,경남은행,1588-8585
4,20250526,0,0,0,0,0,0,0,0,20250601,제주은행,1588-0079
5,20250526,4.04,0,0,0,0,0,0,0,20250601,광주은행,1600-4000
6,20250526,4.8,0,0,0,0,0,0,0,20250601,전북은행,1588-4477
7,20250526,3.37,0,0,0,0,0,0,0,20250601,토스뱅크,1661-7654
8,20250526,2.92,0,0,0,0,0,0,0,20250601,아이엠뱅크,1588-5050
9,20250526,3.93,0,0,0,0,0,0,0,20250601,우리은행,1599-5000


# xml로 가져오기

In [5]:
from bs4 import BeautifulSoup as bs

In [11]:
url="http://apis.data.go.kr/B551408/rent-loan-rate-info/rate-list"
servicekey="TUszyF9E+ahKSr7EHb1uL3WyU458jqCE99oqTvyRDyQ9+oN7KcVAIijXzvc8iFsJhgTjoKHuRWT2wV35QeNBZw=="
payload=dict(serviceKey=servicekey, numOfRows=1000, pageNo=1, dataType='XML')
r=requests.get(url, params=payload)
response=r.content
soup=bs(response, 'xml')
xml_result={}
for item in soup.select('item'):
    for tags in item:
        if tags.name==None:
            continue
        else:
            xml_result.setdefault(tags.name, []).append(tags.text)
xml_result_df=pd.DataFrame(xml_result)
xml_result_df

,bssYmdStart,interest4_1,interest3_2,interest4_2,interest2_1,interest1_2,interest3_1,interest2_2,interest1_1,bssYmdEnd,organId,callCenter
0,20250526,0,0,0,0,0,0,0,0,20250601,산업은행,1588-1500
1,20250526,0,0,0,0,0,0,0,0,20250601,수협은행,1588-1515
2,20250526,0,0,0,0,0,0,0,0,20250601,SC은행,1588-1599
3,20250526,5.06,0,0,0,0,0,0,0,20250601,경남은행,1588-8585
4,20250526,0,0,0,0,0,0,0,0,20250601,제주은행,1588-0079
5,20250526,4.04,0,0,0,0,0,0,0,20250601,광주은행,1600-4000
6,20250526,4.8,0,0,0,0,0,0,0,20250601,전북은행,1588-4477
7,20250526,3.37,0,0,0,0,0,0,0,20250601,토스뱅크,1661-7654
8,20250526,2.92,0,0,0,0,0,0,0,20250601,아이엠뱅크,1588-5050
9,20250526,3.93,0,0,0,0,0,0,0,20250601,우리은행,1599-5000


setdefault를 이용해 xml의 tag명의 유일값 추출하기

In [7]:
tag_names={}
for tags in soup.select("*"):
    tag_names.setdefault(tags.name,"")
for tag_name in tag_names.keys():
    print(tag_name)

response
header
resultCode
resultMsg
body
pageNo
totalCount
numOfRows
items
item
bssYmdStart
interest4_1
interest3_2
interest4_2
interest2_1
interest1_2
interest3_1
interest2_2
interest1_1
bssYmdEnd
organId
callCenter


* xml로 데이터를 받으면 처음에는 단순 문자열로 받게 된다.
* 문자열을 xml 문서로 변환해야 함
* beautifulsoup을 이용해 변환
* beautifulsoup의 메서드인 select, select_one을 이용해 css 셀렉터 기반으로
데이터가 위치한 태그를 찾아서 

# BeautifulSoup
* pip install beautifulsoup4
* from bs4 import BeautifulSoup as bs
* find, find_all 함수: xml, htmlm에서 태그 기반으로 내용을 찾음
* select, select_one 함수: xml, html에서 css selector 기반으로 내용을 찾음
* find_all, select는 해당 태그나 css selector를 가진 부분을 모두 찾아서 list로 반환
* find, select_one은 여러 태그나 css selector 중에서 가장 먼저 나오는 태그/selector를 한개만 찾아줌
* 찾아온 태그 .name: 태그 이름 반환
* 찾아온 태그 .text, .string: 태그 안쪽의 텍스트 반환

In [18]:
from bs4 import BeautifulSoup as bs

In [19]:
soup=bs(response, 'xml')
soup

<?xml version="1.0" encoding="utf-8"?>
<response><header><resultCode>00</resultCode><resultMsg>정상</resultMsg></header><body><pageNo>1</pageNo><totalCount>18</totalCount><numOfRows>10</numOfRows><items><item><bssYmdStart>20250526</bssYmdStart><interest4_1>0</interest4_1><interest3_2>0</interest3_2><interest4_2>0</interest4_2><interest2_1>0</interest2_1><interest1_2>0</interest1_2><interest3_1>0</interest3_1><interest2_2>0</interest2_2><interest1_1>0</interest1_1><bssYmdEnd>20250601</bssYmdEnd><organId>산업은행</organId><callCenter>1588-1500</callCenter></item><item><bssYmdStart>20250526</bssYmdStart><interest4_1>0</interest4_1><interest3_2>0</interest3_2><interest4_2>0</interest4_2><interest2_1>0</interest2_1><interest1_2>0</interest1_2><interest3_1>0</interest3_1><interest2_2>0</interest2_2><interest1_1>0</interest1_1><bssYmdEnd>20250601</bssYmdEnd><organId>수협은행</organId><callCenter>1588-1515</callCenter></item><item><bssYmdStart>20250526</bssYmdStart><interest4_1>0</interest4_1><interest3

In [9]:
xml_result_df

<item><bssYmdStart>20250526</bssYmdStart><interest4_1>0</interest4_1><interest3_2>0</interest3_2><interest4_2>0</interest4_2><interest2_1>0</interest2_1><interest1_2>0</interest1_2><interest3_1>0</interest3_1><interest2_2>0</interest2_2><interest1_1>0</interest1_1><bssYmdEnd>20250601</bssYmdEnd><organId>산업은행</organId><callCenter>1588-1500</callCenter></item>

<item><bssYmdStart>20250526</bssYmdStart><interest4_1>0</interest4_1><interest3_2>0</interest3_2><interest4_2>0</interest4_2><interest2_1>0</interest2_1><interest1_2>0</interest1_2><interest3_1>0</interest3_1><interest2_2>0</interest2_2><interest1_1>0</interest1_1><bssYmdEnd>20250601</bssYmdEnd><organId>수협은행</organId><callCenter>1588-1515</callCenter></item>

<item><bssYmdStart>20250526</bssYmdStart><interest4_1>0</interest4_1><interest3_2>0</interest3_2><interest4_2>0</interest4_2><interest2_1>0</interest2_1><interest1_2>0</interest1_2><interest3_1>0</interest3_1><interest2_2>0</interest2_2><interest1_1>0</interest1_1><bssYmdEnd>

In [20]:
soup.select_one('resultCode')

<resultCode>00</resultCode>

In [25]:
for tag in items.select_one("item"):
    print(tag.name)

bssYmdStart
interest4_1
interest3_2
interest4_2
interest2_1
interest1_2
interest3_1
interest2_2
interest1_1
bssYmdEnd
organId
callCenter


In [26]:
items=soup.select_one('items')
bssYmdStart_list=items.select('bssYmdStart')
interest4_1_list=items.select('interest4_1')
interest3_2_list=items.select('interest3_2')
interest4_2_list=items.select('interest4_2')
interest2_1_list=items.select('interest2_1')
interest1_2_list=items.select('interest1_2')
interest3_1_list=items.select('interest3_1')
interest2_2_list=items.select('interest2_2')
interest1_1_list=items.select('interest1_1')
bssYmdEnd_list=items.select('bssYmdEnd')
organId_list=items.select('organId')
callCenter_list=items.select('callCenter')

In [28]:
tag_list=[bssYmdStart_list,interest4_1_list, interest3_2_list,interest4_2_list, interest2_1_list, interest1_2_list, interest3_1_list, interest2_2_list, interest1_1_list, bssYmdEnd_list, organId_list, callCenter_list ]

In [31]:
result={}
for lists in tag_list:
    for item in lists:
        result.setdefault(item.name, []).append(item.text)
result

{'bssYmdStart': ['20250526',
  '20250526',
  '20250526',
  '20250526',
  '20250526',
  '20250526',
  '20250526',
  '20250526',
  '20250526',
  '20250526'],
 'interest4_1': ['0',
  '0',
  '0',
  '5.06',
  '0',
  '4.04',
  '4.8',
  '3.37',
  '2.92',
  '3.93'],
 'interest3_2': ['0', '0', '0', '0', '0', '0', '0', '0', '0', '0'],
 'interest4_2': ['0', '0', '0', '0', '0', '0', '0', '0', '0', '0'],
 'interest2_1': ['0', '0', '0', '0', '0', '0', '0', '0', '0', '0'],
 'interest1_2': ['0', '0', '0', '0', '0', '0', '0', '0', '0', '0'],
 'interest3_1': ['0', '0', '0', '0', '0', '0', '0', '0', '0', '0'],
 'interest2_2': ['0', '0', '0', '0', '0', '0', '0', '0', '0', '0'],
 'interest1_1': ['0', '0', '0', '0', '0', '0', '0', '0', '0', '0'],
 'bssYmdEnd': ['20250601',
  '20250601',
  '20250601',
  '20250601',
  '20250601',
  '20250601',
  '20250601',
  '20250601',
  '20250601',
  '20250601'],
 'organId': ['산업은행',
  '수협은행',
  'SC은행',
  '경남은행',
  '제주은행',
  '광주은행',
  '전북은행',
  '토스뱅크',
  '아이엠뱅크',
  '우리은행']

# 반복문으로 태그를 자동 추출해서 만들기

In [40]:
for tag in soup.select("item")[0]:
    print(tag.name, tag.text)
#     print(key.name, key.text)

bssYmdStart 20250526
interest4_1 0
interest3_2 0
interest4_2 0
interest2_1 0
interest1_2 0
interest3_1 0
interest2_2 0
interest1_1 0
bssYmdEnd 20250601
organId 산업은행
callCenter 1588-1500


In [41]:
result = {}
for tags in soup.select("item"):
    for tag in tags:
        result.setdefault(tag.name, []).append(tag.text)
df = pd.DataFrame(result)
df

,bssYmdStart,interest4_1,interest3_2,interest4_2,interest2_1,interest1_2,interest3_1,interest2_2,interest1_1,bssYmdEnd,organId,callCenter
0,20250526,0,0,0,0,0,0,0,0,20250601,산업은행,1588-1500
1,20250526,0,0,0,0,0,0,0,0,20250601,수협은행,1588-1515
2,20250526,0,0,0,0,0,0,0,0,20250601,SC은행,1588-1599
3,20250526,5.06,0,0,0,0,0,0,0,20250601,경남은행,1588-8585
4,20250526,0,0,0,0,0,0,0,0,20250601,제주은행,1588-0079
5,20250526,4.04,0,0,0,0,0,0,0,20250601,광주은행,1600-4000
6,20250526,4.8,0,0,0,0,0,0,0,20250601,전북은행,1588-4477
7,20250526,3.37,0,0,0,0,0,0,0,20250601,토스뱅크,1661-7654
8,20250526,2.92,0,0,0,0,0,0,0,20250601,아이엠뱅크,1588-5050
9,20250526,3.93,0,0,0,0,0,0,0,20250601,우리은행,1599-5000


In [42]:
soup.select("item")

[<item><bssYmdStart>20250526</bssYmdStart><interest4_1>0</interest4_1><interest3_2>0</interest3_2><interest4_2>0</interest4_2><interest2_1>0</interest2_1><interest1_2>0</interest1_2><interest3_1>0</interest3_1><interest2_2>0</interest2_2><interest1_1>0</interest1_1><bssYmdEnd>20250601</bssYmdEnd><organId>산업은행</organId><callCenter>1588-1500</callCenter></item>,
 <item><bssYmdStart>20250526</bssYmdStart><interest4_1>0</interest4_1><interest3_2>0</interest3_2><interest4_2>0</interest4_2><interest2_1>0</interest2_1><interest1_2>0</interest1_2><interest3_1>0</interest3_1><interest2_2>0</interest2_2><interest1_1>0</interest1_1><bssYmdEnd>20250601</bssYmdEnd><organId>수협은행</organId><callCenter>1588-1515</callCenter></item>,
 <item><bssYmdStart>20250526</bssYmdStart><interest4_1>0</interest4_1><interest3_2>0</interest3_2><interest4_2>0</interest4_2><interest2_1>0</interest2_1><interest1_2>0</interest1_2><interest3_1>0</interest3_1><interest2_2>0</interest2_2><interest1_1>0</interest1_1><bssYmdE

# xml에 있는 모든 key를 추출하기

In [51]:
tag_names = []
for items in soup.select("item"):
    for item in items:
        tag_names.append(item.name)
len(tag_names)

120

# 집합자료형 set: 중복 없는 데이터 집합 만들기

In [50]:
tags=set(tag_names)
tags

{'body',
 'bssYmdEnd',
 'bssYmdStart',
 'callCenter',
 'header',
 'interest1_1',
 'interest1_2',
 'interest2_1',
 'interest2_2',
 'interest3_1',
 'interest3_2',
 'interest4_1',
 'interest4_2',
 'item',
 'items',
 'numOfRows',
 'organId',
 'pageNo',
 'response',
 'resultCode',
 'resultMsg',
 'totalCount'}

In [52]:
import requests
import json
import os
import pandas as pd
import time

In [55]:
url="https://apis.data.go.kr/1160100/service/GetSmallLoanFinanceInstituteInfoService/getOrdinaryFinanceInfo"
servicekey="TUszyF9E+ahKSr7EHb1uL3WyU458jqCE99oqTvyRDyQ9+oN7KcVAIijXzvc8iFsJhgTjoKHuRWT2wV35QeNBZw=="
payload=dict(serviceKey=servicekey, numOfRows=100, pageNo=1, resultType='json')
r=requests.get(url, params=payload)
response=r.json()
response


{'response': {'header': {'resultCode': '00', 'resultMsg': 'NORMAL SERVICE.'},
  'body': {'numOfRows': 100,
   'pageNo': 1,
   'totalCount': 5439,
   'items': {'item': [{'basYm': '202505',
      'snq': '1',
      'finPrdNm': '새희망홀씨Ⅱ',
      'lnLmt': '3500만원',
      'irtCtg': '변동금리',
      'irt': '은행별 상이',
      'maxTotLnTrm': '은행별 상이',
      'maxDfrmTrm': '-',
      'maxRdptTrm': '-',
      'rdptMthd': '원(리)금균등분할상환',
      'usge': '생계',
      'trgt': '근로자',
      'instCtg': '시중은행',
      'ofrInstNm': '14개 취급은행',
      'rsdAreaPamtEqltIstm': '전국',
      'suprTgtDtlCond': '연소득 4천만원 이하 (개인신용평점 하위 100분의 20*이하는 5천만원 이하) *23년 4월 기준 NICE 749점, KCB 700점 이하',
      'age': '없음',
      'incm': '연소득 4천만원 이하 (개인신용평점 하위 100분의 20*이하는 5천만원 이하)',
      'rsdArea': '-',
      'crdtSc': '신용평가회사의 개인신용평점이 하위 100분의 20에 해당하는자',
      'anin': '-',
      'housHoldCnt': '-',
      'housAr': '-',
      'lnTgtHous': '-',
      'rfrcCnpl': '취급은행 콜센터, 서민금융콜센터 (국번없이)1397',
      'grnInst': '-',
      'jnMthd': '14개 취급

In [58]:
result={}
for tag in response['response']['body']['items']['item']:
    for key, value in tag.items():
        result.setdefault(key, []).append(value)
result

{'basYm': ['202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '202505',
  '2025

In [61]:
page=1
result={}
while True:
    url="https://apis.data.go.kr/1160100/service/GetSmallLoanFinanceInstituteInfoService/getOrdinaryFinanceInfo"
    servicekey="TUszyF9E+ahKSr7EHb1uL3WyU458jqCE99oqTvyRDyQ9+oN7KcVAIijXzvc8iFsJhgTjoKHuRWT2wV35QeNBZw=="
    payload=dict(serviceKey=servicekey, numOfRows=100, pageNo=page, resultType='json')
    r=requests.get(url, params=payload)
    response=r.json()
    
    for tag in response['response']['body']['items']['item']:
        for key, value in tag.items():
            result.setdefault(key, []).append(value)
            
    total_pages=int(response['response']['body']['totalCount'])//100+1
    
    if page<total_pages:
        page+=1
    else:
        break
        
    
 
            
df=pd.DataFrame(result)
df
    

,basYm,snq,finPrdNm,lnLmt,irtCtg,irt,maxTotLnTrm,maxDfrmTrm,maxRdptTrm,rdptMthd,...,prdExisYn,cv19Rfrc,prdCtg,prdNm,cv19SuprCtg,cv19SuprCtn,cv19SuprTgtDtlCtn,mgmDln,prdCtg2,fileWrtDt
0,202505,1,새희망홀씨Ⅱ,3500만원,변동금리,은행별 상이,은행별 상이,-,-,원(리)금균등분할상환,...,Y,-,1,대출상품,-,-,-,상시,-,202506010700
1,202505,2,징검다리론,3000만원,변동금리,은행별 상이,5년,1년,4년,원(리)금균등분할상환,...,Y,-,1,대출상품,-,-,-,기관 문의,-,202506010700
2,202505,3,"우리지역 氣-Up 서포트론(영세 소기업,소상공인)","5,00010,000 (코로나 19 피해 소기업, 소상공인)",변동금리,6.27~8.97%,5년,1년,4년,원금균등분할상환,...,Y,-,1,대출상품,-,-,-,상시,-,202506010700
3,202505,4,경상남도 청년 전세자금(경상남도 협약 상품)(경상남도 청년주택 임차보증금 이자지원사업),9000만원,변동금리,은행별 상이,임대차계약기간 이내 최대 2년,0년,0년,일시상환,...,Y,-,1,대출상품,-,-,-,상시,-,202506010700
4,202505,5,i-ONE근로자생활안정자금대출,1000만원,고정금리,1.5 / 2.6 / 1.0 / 3.0,"근로자생활안정자금, 임금체불생계비, 체불근로자생계비 : 2년, 4년, 5년, 6년,...",6년,5년,원(리)금균등분할상환,...,Y,-,1,대출상품,-,-,-,상시,-,202506010700
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5434,202110,425,"온(溫, ON)택트 특례보증",2850만원,변동금리,대출실행 은행에 문의,5년,1년,4년,원금균등분할상환,...,Y,None,1,대출상품,None,None,None,None,None,202502010700
5435,202110,426,위기업종 지원 협약보증,5000만원,-,-,"1, 5년",1년,4년,원금균등분할상환,...,Y,None,1,대출상품,None,None,None,None,None,202502010700
5436,202110,427,익산시 소상공인 특례보증(2023 하나은행 출연),5000만원,고정금리,약 1.6%,5년,1년,4년,원금균등분할상환,...,Y,None,1,대출상품,None,None,None,None,None,202502010700
5437,202110,428,중소벤처기업부 소상공인자금,7000만원,자금별 별도금리 적용,-,5년,2년,3년,분할상환,...,Y,None,1,대출상품,None,None,None,None,None,202502010700


# xml로 가져오기

In [19]:
xml_result_df_list=[]
page=1

while True:
    url="https://apis.data.go.kr/1160100/service/GetSmallLoanFinanceInstituteInfoService/getOrdinaryFinanceInfo"
    servicekey="TUszyF9E+ahKSr7EHb1uL3WyU458jqCE99oqTvyRDyQ9+oN7KcVAIijXzvc8iFsJhgTjoKHuRWT2wV35QeNBZw=="
    payload=dict(serviceKey=servicekey, numOfRows=1000, pageNo=1, resultType='xml')
    r=requests.get(url, params=payload)
    response=r.content
    soup=bs(response, 'xml')
    xml_result={}
    for item in soup.select('item'):
        for tags in item:
            if tags.name==None:
                continue
            else:
                xml_result.setdefault(tags.name,[]).append(tags.text)
    xml_result_df_list.append(pd.DataFrame(xml_result))
    numOfRows=int(soup.select_one('numOfRows').text)
    totalCount=int(soup.select_one('totalCount').text)
    total_pages=totalCount//numOfRows+1
    if page<total_pages:
        page+=1
    else:
        break
        
xml_result_df_list=pd.concat(xml_result_df_list)
xml_result_df_list=xml_result_df_list.reset_index(drop=True)
xml_result_df_list

,basYm,snq,finPrdNm,lnLmt,irtCtg,irt,maxTotLnTrm,maxDfrmTrm,maxRdptTrm,rdptMthd,...,prdExisYn,cv19Rfrc,prdCtg,prdNm,cv19SuprCtg,cv19SuprCtn,cv19SuprTgtDtlCtn,mgmDln,prdCtg2,fileWrtDt
0,202505,1,새희망홀씨Ⅱ,3500만원,변동금리,은행별 상이,은행별 상이,-,-,원(리)금균등분할상환,...,Y,-,1,대출상품,-,-,-,상시,-,202506010700
1,202505,2,징검다리론,3000만원,변동금리,은행별 상이,5년,1년,4년,원(리)금균등분할상환,...,Y,-,1,대출상품,-,-,-,기관 문의,-,202506010700
2,202505,3,"우리지역 氣-Up 서포트론(영세 소기업,소상공인)","5,00010,000 (코로나 19 피해 소기업, 소상공인)",변동금리,6.27~8.97%,5년,1년,4년,원금균등분할상환,...,Y,-,1,대출상품,-,-,-,상시,-,202506010700
3,202505,4,경상남도 청년 전세자금(경상남도 협약 상품)(경상남도 청년주택 임차보증금 이자지원사업),9000만원,변동금리,은행별 상이,임대차계약기간 이내 최대 2년,0년,0년,일시상환,...,Y,-,1,대출상품,-,-,-,상시,-,202506010700
4,202505,5,i-ONE근로자생활안정자금대출,1000만원,고정금리,1.5 / 2.6 / 1.0 / 3.0,"근로자생활안정자금, 임금체불생계비, 체불근로자생계비 : 2년, 4년, 5년, 6년,...",6년,5년,원(리)금균등분할상환,...,Y,-,1,대출상품,-,-,-,상시,-,202506010700
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5995,202503,299,희망전북 함께도~ 특례보증,80000만원,변동금리,~3.5%,1~5,-,-,일시상환,...,Y,-,1,대출상품,-,-,-,기관 문의,-,202504010700
5996,202503,300,익산시 희망더드림 특례보증(Track1 튼튼소상공인),5000만원,변동금리,~2.68%,1~5,1년,"4, 5년",원금균등분할상환,...,Y,-,1,대출상품,-,-,-,기관 문의,-,202504010700
5997,202503,301,익산시 희망더드림 특례보증(Track2 탄탄소상공인),5000만원,변동금리,~1.68%,1~5,1년,"4, 5년",원금균등분할상환,...,Y,-,1,대출상품,-,-,-,기관 문의,-,202504010700
5998,202503,302,전주시 희망더드림 안심연장 특례보증,-,변동금리,~ 2.48%,9년,1년,8년,원금균등분할상환,...,Y,-,1,대출상품,-,-,-,기관 문의,-,202504010700


In [18]:
xml_result_df_list = []
page = 1
while True:
    url = "https://apis.data.go.kr/1160100/service/GetSmallLoanFinanceInstituteInfoService/getOrdinaryFinanceInfo"
    service_key = "TUszyF9E+ahKSr7EHb1uL3WyU458jqCE99oqTvyRDyQ9+oN7KcVAIijXzvc8iFsJhgTjoKHuRWT2wV35QeNBZw=="
    payload = dict(serviceKey=service_key, numOfRows=1000, pageNo=page, resultType='xml')
    r = requests.get(url, params=payload)
    print(r.url)
    print(r.status_code)
    response = r.content
    soup = bs(response, "xml")
    xml_result = {}
    for item in soup.select("item"):
        for tags in item:
            if tags.name == None:
                continue
            else:
                xml_result.setdefault(tags.name, []).append(tags.text)
    xml_result_df_list.append(pd.DataFrame(xml_result))
    numOfRows = int(soup.select_one("numOfRows").text)
    totalCount = int(soup.select_one("totalCount").text)
    total_pages = totalCount // numOfRows + 1
    if page < total_pages:
        page += 1
    else:
        break
        
xml_result_df_list = pd.concat(xml_result_df_list)
xml_result_df_list = xml_result_df_list.reset_index(drop=True)
xml_result_df_list

https://apis.data.go.kr/1160100/service/GetSmallLoanFinanceInstituteInfoService/getOrdinaryFinanceInfo?serviceKey=TUszyF9E%2BahKSr7EHb1uL3WyU458jqCE99oqTvyRDyQ9%2BoN7KcVAIijXzvc8iFsJhgTjoKHuRWT2wV35QeNBZw%3D%3D&numOfRows=1000&pageNo=1&resultType=xml
200
https://apis.data.go.kr/1160100/service/GetSmallLoanFinanceInstituteInfoService/getOrdinaryFinanceInfo?serviceKey=TUszyF9E%2BahKSr7EHb1uL3WyU458jqCE99oqTvyRDyQ9%2BoN7KcVAIijXzvc8iFsJhgTjoKHuRWT2wV35QeNBZw%3D%3D&numOfRows=1000&pageNo=2&resultType=xml
200
https://apis.data.go.kr/1160100/service/GetSmallLoanFinanceInstituteInfoService/getOrdinaryFinanceInfo?serviceKey=TUszyF9E%2BahKSr7EHb1uL3WyU458jqCE99oqTvyRDyQ9%2BoN7KcVAIijXzvc8iFsJhgTjoKHuRWT2wV35QeNBZw%3D%3D&numOfRows=1000&pageNo=3&resultType=xml
200
https://apis.data.go.kr/1160100/service/GetSmallLoanFinanceInstituteInfoService/getOrdinaryFinanceInfo?serviceKey=TUszyF9E%2BahKSr7EHb1uL3WyU458jqCE99oqTvyRDyQ9%2BoN7KcVAIijXzvc8iFsJhgTjoKHuRWT2wV35QeNBZw%3D%3D&numOfRows=1000&pageNo=4&res

,basYm,snq,finPrdNm,lnLmt,irtCtg,irt,maxTotLnTrm,maxDfrmTrm,maxRdptTrm,rdptMthd,...,prdExisYn,cv19Rfrc,prdCtg,prdNm,cv19SuprCtg,cv19SuprCtn,cv19SuprTgtDtlCtn,mgmDln,prdCtg2,fileWrtDt
0,202505,1,새희망홀씨Ⅱ,3500만원,변동금리,은행별 상이,은행별 상이,-,-,원(리)금균등분할상환,...,Y,-,1,대출상품,-,-,-,상시,-,202506010700
1,202505,2,징검다리론,3000만원,변동금리,은행별 상이,5년,1년,4년,원(리)금균등분할상환,...,Y,-,1,대출상품,-,-,-,기관 문의,-,202506010700
2,202505,3,"우리지역 氣-Up 서포트론(영세 소기업,소상공인)","5,00010,000 (코로나 19 피해 소기업, 소상공인)",변동금리,6.27~8.97%,5년,1년,4년,원금균등분할상환,...,Y,-,1,대출상품,-,-,-,상시,-,202506010700
3,202505,4,경상남도 청년 전세자금(경상남도 협약 상품)(경상남도 청년주택 임차보증금 이자지원사업),9000만원,변동금리,은행별 상이,임대차계약기간 이내 최대 2년,0년,0년,일시상환,...,Y,-,1,대출상품,-,-,-,상시,-,202506010700
4,202505,5,i-ONE근로자생활안정자금대출,1000만원,고정금리,1.5 / 2.6 / 1.0 / 3.0,"근로자생활안정자금, 임금체불생계비, 체불근로자생계비 : 2년, 4년, 5년, 6년,...",6년,5년,원(리)금균등분할상환,...,Y,-,1,대출상품,-,-,-,상시,-,202506010700
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5434,202110,425,"온(溫, ON)택트 특례보증",2850만원,변동금리,대출실행 은행에 문의,5년,1년,4년,원금균등분할상환,...,Y,,1,대출상품,,,,,,202502010700
5435,202110,426,위기업종 지원 협약보증,5000만원,-,-,"1, 5년",1년,4년,원금균등분할상환,...,Y,,1,대출상품,,,,,,202502010700
5436,202110,427,익산시 소상공인 특례보증(2023 하나은행 출연),5000만원,고정금리,약 1.6%,5년,1년,4년,원금균등분할상환,...,Y,,1,대출상품,,,,,,202502010700
5437,202110,428,중소벤처기업부 소상공인자금,7000만원,자금별 별도금리 적용,-,5년,2년,3년,분할상환,...,Y,,1,대출상품,,,,,,202502010700


In [21]:
key_list={}
for item in soup.select("*"):
    key_list.setdefault(item.name,"")
key_list=list(key_list.keys())
key_list

['response',
 'header',
 'resultCode',
 'resultMsg',
 'body',
 'numOfRows',
 'pageNo',
 'totalCount',
 'items',
 'item',
 'basYm',
 'snq',
 'finPrdNm',
 'lnLmt',
 'irtCtg',
 'irt',
 'maxTotLnTrm',
 'maxDfrmTrm',
 'maxRdptTrm',
 'rdptMthd',
 'usge',
 'trgt',
 'instCtg',
 'ofrInstNm',
 'rsdAreaPamtEqltIstm',
 'suprTgtDtlCond',
 'age',
 'incm',
 'rsdArea',
 'crdtSc',
 'anin',
 'housHoldCnt',
 'housAr',
 'lnTgtHous',
 'rfrcCnpl',
 'grnInst',
 'jnMthd',
 'rpymdCfe',
 'lnIcdcst',
 'ovItrYr',
 'prftAddIrtCond',
 'etcRefSbjc',
 'hdlInst',
 'cnpl',
 'rltSite',
 'tgtFltr',
 'hdlInstDtlVw',
 'prdExisYn',
 'cv19Rfrc',
 'prdCtg',
 'prdNm',
 'cv19SuprCtg',
 'cv19SuprCtn',
 'cv19SuprTgtDtlCtn',
 'mgmDln',
 'prdCtg2',
 'fileWrtDt']

In [22]:
keys=key_list[10:]

In [23]:
soup.select('basYm')
soup.select("snq")

[<snq>1</snq>,
 <snq>2</snq>,
 <snq>3</snq>,
 <snq>4</snq>,
 <snq>5</snq>,
 <snq>6</snq>,
 <snq>7</snq>,
 <snq>8</snq>,
 <snq>9</snq>,
 <snq>10</snq>,
 <snq>11</snq>,
 <snq>12</snq>,
 <snq>13</snq>,
 <snq>14</snq>,
 <snq>15</snq>,
 <snq>16</snq>,
 <snq>17</snq>,
 <snq>18</snq>,
 <snq>19</snq>,
 <snq>20</snq>,
 <snq>21</snq>,
 <snq>22</snq>,
 <snq>23</snq>,
 <snq>24</snq>,
 <snq>25</snq>,
 <snq>26</snq>,
 <snq>27</snq>,
 <snq>28</snq>,
 <snq>29</snq>,
 <snq>30</snq>,
 <snq>31</snq>,
 <snq>33</snq>,
 <snq>34</snq>,
 <snq>35</snq>,
 <snq>36</snq>,
 <snq>37</snq>,
 <snq>38</snq>,
 <snq>39</snq>,
 <snq>40</snq>,
 <snq>41</snq>,
 <snq>42</snq>,
 <snq>43</snq>,
 <snq>44</snq>,
 <snq>45</snq>,
 <snq>46</snq>,
 <snq>47</snq>,
 <snq>48</snq>,
 <snq>49</snq>,
 <snq>50</snq>,
 <snq>51</snq>,
 <snq>52</snq>,
 <snq>53</snq>,
 <snq>54</snq>,
 <snq>55</snq>,
 <snq>56</snq>,
 <snq>57</snq>,
 <snq>58</snq>,
 <snq>59</snq>,
 <snq>60</snq>,
 <snq>61</snq>,
 <snq>62</snq>,
 <snq>63</snq>,
 <snq>64</snq>,
 

In [24]:
result={}
for key in keys:
    for item in soup.select(key):
        result.setdefault(item.name, []).append(item.text)
df=pd.DataFrame(result)
df

basYm
snq
finPrdNm
lnLmt
irtCtg
irt
maxTotLnTrm
maxDfrmTrm
maxRdptTrm
rdptMthd
usge
trgt
instCtg
ofrInstNm
rsdAreaPamtEqltIstm
suprTgtDtlCond
age
incm
rsdArea
crdtSc
anin
housHoldCnt
housAr
lnTgtHous
rfrcCnpl
grnInst
jnMthd
rpymdCfe
lnIcdcst
ovItrYr
prftAddIrtCond
etcRefSbjc
hdlInst
cnpl
rltSite
tgtFltr
hdlInstDtlVw
prdExisYn
cv19Rfrc
prdCtg
prdNm
cv19SuprCtg
cv19SuprCtn
cv19SuprTgtDtlCtn
mgmDln
prdCtg2
fileWrtDt


# key_list를 만들고 select로 직접 추출하기

In [25]:
df_list = []
page = 1
while True:
    url = "https://apis.data.go.kr/1160100/service/GetSmallLoanFinanceInstituteInfoService/getOrdinaryFinanceInfo"
    service_key = "8Ym5dhmVJdr12XzGnyYrQjSBS1QuRBYK8yHfx65JCl4vACM9uLKo8fxCVOFJkPB71llD7F2rOEROZnHgNGoj3A=="
    payload = dict(serviceKey=service_key, numOfRows=1000, pageNo=page, resultType='xml')
    r = requests.get(url, params=payload)
    print(r.url)
    print(r.status_code)
    response = r.content
    soup = bs(response, "xml")
    numOfRows = int(soup.select_one("numOfRows").text)
    totalCount = int(soup.select_one("totalCount").text)
    total_pages = totalCount // numOfRows + 1
    key_list = {}
    for item in soup.select("*"):
        key_list.setdefault(item.name, "")
    keys = list(key_list.keys())
    result = {}
    for key in keys[10:]:
        for item in soup.select(key):
            result.setdefault(item.name, []).append(item.text)
    df_list.append(pd.DataFrame(result))
    if page < total_pages:
        page += 1
    else:
        print("데이터 수집완료")
        break
result_df = pd.concat(df_list)
result_df = result_df.reset_index(drop=True)
result_df

https://apis.data.go.kr/1160100/service/GetSmallLoanFinanceInstituteInfoService/getOrdinaryFinanceInfo?serviceKey=8Ym5dhmVJdr12XzGnyYrQjSBS1QuRBYK8yHfx65JCl4vACM9uLKo8fxCVOFJkPB71llD7F2rOEROZnHgNGoj3A%3D%3D&numOfRows=1000&pageNo=1&resultType=xml
200
https://apis.data.go.kr/1160100/service/GetSmallLoanFinanceInstituteInfoService/getOrdinaryFinanceInfo?serviceKey=8Ym5dhmVJdr12XzGnyYrQjSBS1QuRBYK8yHfx65JCl4vACM9uLKo8fxCVOFJkPB71llD7F2rOEROZnHgNGoj3A%3D%3D&numOfRows=1000&pageNo=2&resultType=xml
200
https://apis.data.go.kr/1160100/service/GetSmallLoanFinanceInstituteInfoService/getOrdinaryFinanceInfo?serviceKey=8Ym5dhmVJdr12XzGnyYrQjSBS1QuRBYK8yHfx65JCl4vACM9uLKo8fxCVOFJkPB71llD7F2rOEROZnHgNGoj3A%3D%3D&numOfRows=1000&pageNo=3&resultType=xml
200
https://apis.data.go.kr/1160100/service/GetSmallLoanFinanceInstituteInfoService/getOrdinaryFinanceInfo?serviceKey=8Ym5dhmVJdr12XzGnyYrQjSBS1QuRBYK8yHfx65JCl4vACM9uLKo8fxCVOFJkPB71llD7F2rOEROZnHgNGoj3A%3D%3D&numOfRows=1000&pageNo=4&resultType=xml
200


,basYm,snq,finPrdNm,lnLmt,irtCtg,irt,maxTotLnTrm,maxDfrmTrm,maxRdptTrm,rdptMthd,...,prdExisYn,cv19Rfrc,prdCtg,prdNm,cv19SuprCtg,cv19SuprCtn,cv19SuprTgtDtlCtn,mgmDln,prdCtg2,fileWrtDt
0,202505,1,새희망홀씨Ⅱ,3500만원,변동금리,은행별 상이,은행별 상이,-,-,원(리)금균등분할상환,...,Y,-,1,대출상품,-,-,-,상시,-,202506010700
1,202505,2,징검다리론,3000만원,변동금리,은행별 상이,5년,1년,4년,원(리)금균등분할상환,...,Y,-,1,대출상품,-,-,-,기관 문의,-,202506010700
2,202505,3,"우리지역 氣-Up 서포트론(영세 소기업,소상공인)","5,00010,000 (코로나 19 피해 소기업, 소상공인)",변동금리,6.27~8.97%,5년,1년,4년,원금균등분할상환,...,Y,-,1,대출상품,-,-,-,상시,-,202506010700
3,202505,4,경상남도 청년 전세자금(경상남도 협약 상품)(경상남도 청년주택 임차보증금 이자지원사업),9000만원,변동금리,은행별 상이,임대차계약기간 이내 최대 2년,0년,0년,일시상환,...,Y,-,1,대출상품,-,-,-,상시,-,202506010700
4,202505,5,i-ONE근로자생활안정자금대출,1000만원,고정금리,1.5 / 2.6 / 1.0 / 3.0,"근로자생활안정자금, 임금체불생계비, 체불근로자생계비 : 2년, 4년, 5년, 6년,...",6년,5년,원(리)금균등분할상환,...,Y,-,1,대출상품,-,-,-,상시,-,202506010700
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5434,202110,425,"온(溫, ON)택트 특례보증",2850만원,변동금리,대출실행 은행에 문의,5년,1년,4년,원금균등분할상환,...,Y,,1,대출상품,,,,,,202502010700
5435,202110,426,위기업종 지원 협약보증,5000만원,-,-,"1, 5년",1년,4년,원금균등분할상환,...,Y,,1,대출상품,,,,,,202502010700
5436,202110,427,익산시 소상공인 특례보증(2023 하나은행 출연),5000만원,고정금리,약 1.6%,5년,1년,4년,원금균등분할상환,...,Y,,1,대출상품,,,,,,202502010700
5437,202110,428,중소벤처기업부 소상공인자금,7000만원,자금별 별도금리 적용,-,5년,2년,3년,분할상환,...,Y,,1,대출상품,,,,,,202502010700
